#### Simple MLP to test whether MBE regularization could "resolve conflict"

In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import DataLoader, TensorDataset

from src.rank_regularizer import patch_mbe3
from src.pcgrad import YetAnotherMixer2

# utils function
# ------------------------------------------------------------
def generate_param_shift(model, direction_vector=None):
    direction_vector = {}
    for name, param in model.named_parameters():
        if 'weight' in name:  # Focus on weight matrices
            if len(param.shape) > 1:
                rank = min(3, min(param.shape))
                u = torch.randn(param.shape[0], rank, device=param.device)
                v = torch.randn(rank, param.shape[1], device=param.device)
                dir_tensor = torch.matmul(u, v)
                dir_tensor = dir_tensor / dir_tensor.norm() * param.norm()
            else:
                dir_tensor = torch.randn_like(param)
                dir_tensor = dir_tensor / dir_tensor.norm() * param.norm()
            direction_vector[name] = dir_tensor
    return direction_vector

def apply_param_shift(model, direction_vector, magnitude=0.1): 
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in direction_vector:
                param.add_(direction_vector[name] * magnitude)
    return model 

def build_dataset(train_size, val_size):
    data_size = train_size + val_size 
    x = torch.randn(data_size, in_dim)
    param_shift = generate_param_shift(mlp)
    mlp_positive_shift = apply_param_shift(mlp, param_shift, 0.8)
    y_positive, h_positive = mlp_positive_shift(x)
    mlp_negative_shift = apply_param_shift(mlp, param_shift, -0.8)
    y_negative, h_negative = mlp_negative_shift(x)
    print(f"- Dataset constructed with {data_size} positive & negative samples")
    trainset = {"positive": (x[:train_size], y_positive.detach()[:train_size]), "negative": (x[:train_size], y_negative.detach()[:train_size])}
    valset = {"positive": (x[train_size:], y_positive.detach()[train_size:]), "negative": (x[train_size:], y_negative.detach()[train_size:])}
    return trainset, valset, param_shift

def get_batch(dataset, group_name, batch_size=32): 
    x, y = dataset[group_name]
    indices = torch.randperm(len(x))
    batch_indices = indices[:batch_size]
    return x[batch_indices], y[batch_indices]


# MBE & L1 loss
# ------------------------------------------------------------
def mbe_loss(x, patch_size=8):
    return patch_mbe3(x.unsqueeze(0), patch_size)

def l1_loss(y_pred, y): 
    return torch.nn.functional.l1_loss(y_pred, y)

# Simple MLP model
# ------------------------------------------------------------
class SimpleModel(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=50, output_dim=1):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        h = self.activation(self.layer1(x))
        return self.layer2(h), h
    
    def compute_loss(self, x, y): 
        y_pred, h = self(x)
        loss_dict = {"l1": l1_loss(y_pred, y), "mbe": mbe_loss(h)}
        return loss_dict
        
    def get_hidden_representation(self, x):
        return self.activation(self.layer1(x))

# Hyper-param     
# ------------------------------------------------------------
in_dim, hidden_dim, out_dim = 10, 50, 1 
data_size = 1000 

# Model initialization 
# ------------------------------------------------------------
mlp = SimpleModel(in_dim, hidden_dim, out_dim)

# Gradient composer 
# ------------------------------------------------------------
grad_mixer = YetAnotherMixer2(mlp, "l1")

# Build 'Conflicting' dataset
# ------------------------------------------------------------
trainset, valset, param_shift = build_dataset(500, 100)

# Different Learning approaches 
# ------------------------------------------------------------
# (a). Train with P samples, then N samples 
# Initialize optimizer
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.01)
epochs = 10
batch_size = 32
train_accumulation_steps = 16
val_interval = 5
val_steps = 10


# Phase manager 
# ------------------------------------------------------------
from src.utils import RRScheduler
scheduler = RRScheduler(train_accumulation_steps, epochs,
                        start_layer = 0, 
                        end_layer = 1,
                        main_loss_name = "l1", 
                        switch_phase = True, 
                        entropy_patience=25,
                        mbe_patience=25,
                        inv_mbe_patience=10,
                        include_inner_cycle=False)
# ------------------------------------------------------------


# Train with positive samples first, then negative samples
print("Training with positive samples first, then negative samples")

loss_record = defaultdict(list)

# Positive samples training
for epoch in range(epochs):
    group_name = "positive"
    if epoch % val_interval == 0: 
        mlp.eval() 
        val_loss = defaultdict(float)
        with torch.no_grad():
            for i in range(val_steps):
                inputs, targets = get_batch(valset, group_name)
                loss_dict = mlp.compute_loss(inputs, targets)
                for name, loss in loss_dict.items(): 
                    val_loss[name] += loss                
        for name in val_loss: 
            val_loss[name] /= val_steps
            loss_record[name].append(val_loss[name].item())
        mlp.train() 
        continue
    print(f" Epoch {epoch+1}/{epochs}, {group_name} l1 loss: {val_loss['l1']:.4f} | mbe loss: {val_loss['mbe']:.4f}")
    
    for accum_step in range(train_accumulation_steps): 
        x_batch, y_batch = get_batch(trainset, group_name)
        optimizer.zero_grad()
        loss_dict = mlp.compute_loss(x_batch, y_batch)
        if accum_step == train_accumulation_steps - 1: 
            scheduler.step(loss_dict)
        
        if  len(scheduler.rr_layer_index) == 0: 
            loss_dict = {"l1": loss_dict["l1"]}
            print(f"- backward on l1 loss -")
        else:        
            print(f"- backward on mbe loss -")
            loss_dict = {"mbe": loss_dict["mbe"] * scheduler.rr_layer_weight}
        
        grad_mixer.naive_backward_info(loss_dict)
    optimizer.step()
    
    break
    


- Dataset constructed with 600 positive & negative samples
Training with positive samples first, then negative samples
 Epoch 2/10, positive l1 loss: 0.2432 | mbe loss: 1.1578
- backward on l1 loss -
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
- backward on l1 loss -
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
- backward on l1 loss -
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
- backward on l1 loss -
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
- backward on l1 loss -
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
- backward on l1 loss -
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
 None Gradient encountered
- backward on l1 loss -
 None Gr

In [37]:
# Current schedule : memorize --> decrease MBE --> memorize --> decrease MBE --> ... (if memory decays in MBE decrease, get into memorization phase)
# Cyclic schedule : increase MBE --> memorize --> decrease MBE (2 beats) --> memorize --> increase MBE --> ..... 

896

In [ ]:
# Get weight in MLP param 

# Sample a bunch of input samples 

# 1. Data preparation 
# Design a "shift" in the param space 
# - Apply shift to param, use resulting model to get output (targets) for all inputs 
# - Revert that 'shift' (Or just rotate it with near 180 degree) to obtain 'negative samples'
# This would give us a pair of datasets, which is nagatively aligned with each other in its 'loss's gradient over parameter'
# loss should be simple L1 loss

# 2. Training
# - Train with A, then B. 
# - Train with B, then A. 
# - Interleaved Training (A->B->A->B... or even just mix A & B together etc ...)
# - PGS-enhanced MBE regularization (Algo 3.a)
# - PBRS (Algo 3.c)
# - Phase transition (Algo 3.b)


# 3. Evaluation
# 1. L1 loss
# 2. MBE loss 
# 3. Gradient conflicts? (Intermediate gradient conflict, if we could observe it after 'resolving conflicting' in compression phase would be nice)
